<a href="https://colab.research.google.com/github/Quratulain-12/Bioinformatic-services/blob/main/fetch_fasta_integrated_wgs_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import time
from urllib.error import HTTPError

# 1. Load the raw text matrix file
# Adjust 'New Text Document.txt' if your file name or path is different
try:
    df = pd.read_csv("/content/final.txt", sep="\t")
except Exception:
    df = pd.read_csv("/content/final.txt", sep=" ")

# 2. Filter for core genes: drop rows where any of the 6 strain columns have missing values
core_df = df.dropna().reset_index(drop=True)
print(f"Total core genes identified: {len(core_df)}")

# 3. Extract accession IDs for the first strain to use as our reference sequence set
accessions = core_df["Acinetobacter_baumannii_DT_Ab057"].tolist()

# 4. Use Biopython to fetch sequences directly from NCBI Entrez safely
try:
    from Bio import Entrez
except ImportError:
    import os
    os.system('pip install biopython')
    from Bio import Entrez

# Provide your email to NCBI
Entrez.email = "your.email@university.edu"

fasta_output = "final_core_proteins.fasta"
batch_size = 100  # Small batch sizes to prevent NCBI from hanging
print("Connecting to NCBI Entrez Server... Fetching sequences...")

with open(fasta_output, "w") as out_file:
    for i in range(0, len(accessions), batch_size):
        batch = accessions[i:i+batch_size]
        id_list = ",".join(batch)

        success = False
        retries = 3
        while not success and retries > 0:
            try:
                handle = Entrez.efetch(db="protein", id=id_list, rettype="fasta", retmode="text")
                data = handle.read()
                out_file.write(data)
                handle.close()
                success = True
                print(f"Successfully fetched records {i+1} to {min(i+batch_size, len(accessions))}")
                time.sleep(1)  # Respect NCBI server guidelines
            except HTTPError as e:
                print(f"Server busy, retrying in 5 seconds... (Retries left: {retries})")
                retries -= 1
                time.sleep(5)

print(f"\nSuccess! Your file '{fasta_output}' has been created with all core protein sequences.")

Total core genes identified: 2828
Connecting to NCBI Entrez Server... Fetching sequences...
Successfully fetched records 1 to 100
Successfully fetched records 101 to 200
Successfully fetched records 201 to 300
Successfully fetched records 301 to 400
Successfully fetched records 401 to 500
Successfully fetched records 501 to 600
Successfully fetched records 601 to 700
Successfully fetched records 701 to 800
Successfully fetched records 801 to 900
Successfully fetched records 901 to 1000
Successfully fetched records 1001 to 1100
Successfully fetched records 1101 to 1200
Successfully fetched records 1201 to 1300
Successfully fetched records 1301 to 1400
Successfully fetched records 1401 to 1500
Successfully fetched records 1501 to 1600
Successfully fetched records 1601 to 1700
Successfully fetched records 1701 to 1800
Successfully fetched records 1801 to 1900
Successfully fetched records 1901 to 2000
Successfully fetched records 2001 to 2100
Successfully fetched records 2101 to 2200
Succe